O objetivo é criar uma consulta de ações no yahoo finance com o objetivo de armazenar as informações em um banco de dados local, criando um histórico de consulta que pode ser utilizado para análises futuras e que poderá ser atualizado periodicamente para manter os dados atualizados. Para isso, podemos utilizar a biblioteca `yfinance` em Python, que facilita a obtenção de dados financeiros do Yahoo Finance.



In [1]:
import yfinance as yf
from datetime import datetime, timedelta
import pandas as pd


### Datas das consultas

In [2]:
# Data de início para coleta de dados
data_marco = '2023-01-01'

# Data atual para a coleta de dados
data_atual = datetime.now().strftime('%Y-%m-%d')

### Tratamento da consulta de ações (tickers) para transformar em um json para consulta

#### Função para carregar dados de empresas e setores da B3

In [3]:
import pandas as pd

def gerar_df_b3_empresas_setor(caminho_arquivo: str, 
                                    planilha: str = 'Setor', 
                                    colunas_intervalo: str = 'B:H', 
                                    pula_linhas: int = 2, 
                                    numero_linhas: int = 370) -> pd.DataFrame:
    """
    Carrega e retorna um DataFrame com as informações de empresas e setores da B3.
    
    Esta função lê um arquivo Excel contendo dados de empresas listadas na Bolsa
    de Valores do Brasil (B3), com informações sobre setores econômicos,
    subsetores, segmentos de negociação e outros dados estruturais.
    
    Parâmetros
    ----------
    caminho_arquivo : str
        Caminho completo ou relativo do arquivo Excel contendo dados das empresas e setores.
        Exemplo: '../utils/gera_tikers/B3_Empresas_Setor_20260206.xlsx'
    
    planilha : str, optional
        Nome da aba/planilha no arquivo Excel (padrão: 'Setor').
        A planilha deve conter uma coluna chamada 'CÓDIGO' que será usada como índice.
    
    colunas_intervalo : str, optional
        Intervalo de colunas a ler no formato Excel (padrão: 'B:H').
        Exemplos válidos: 'A:E', 'B:H', 'C:F'.
    
    pula_linhas : int, optional
        Número de linhas iniciais a pular na leitura (padrão: 2).
        Útil para ignorar cabeçalhos ou informações adicionais no início do arquivo.
    
    numero_linhas : int, optional
        Número máximo de linhas de dados a ler (padrão: 370).
        Se o arquivo tiver menos linhas, todas serão lidas.
    
    Retorno
    -------
    pd.DataFrame
        DataFrame com a coluna 'CÓDIGO' definida como índice, contendo as seguintes
        informações estruturais:
        - BEEST (ou similar)
        - SETOR ECONÔMICO
        - SUBSETOR
        - SEGMENTO
        - NOME DE PREGÃO
        - SEGMENTO DE NEGOCIAÇÃO
        
        O índice do DataFrame conterá os códigos das empresas (ex: 'PETR', 'VALE', 'ITUB').
    """
    try:
        # Ler o arquivo Excel com os parâmetros especificados
        df_b3_empresas_setor = pd.read_excel(
            io=caminho_arquivo,
            sheet_name=planilha,
            usecols=colunas_intervalo,
            skiprows=pula_linhas,
            nrows=numero_linhas,
            index_col='CÓDIGO'  # Definir a coluna 'CÓDIGO' como índice do DataFrame
        )
        
        return df_b3_empresas_setor
    
    # Tratamento de exceções para garantir que erros sejam informativos
    except FileNotFoundError:
        raise FileNotFoundError(f"Arquivo não encontrado: {caminho_arquivo}")
    except ValueError as e:
        raise ValueError(f"Erro ao ler o arquivo Excel: {str(e)}")
    except Exception as e:
        raise Exception(f"Erro inesperado ao processar o arquivo: {str(e)}")


In [4]:
# Exemplo de uso: Carregar o DataFrame de empresas e setores da B3
caminho_arquivo_b3 = "../utils/gera_tikers/B3_Empresas_Setor_20260206.xlsx"

df_b3_empresas_setor = gerar_df_b3_empresas_setor(
    caminho_arquivo=caminho_arquivo_b3,
    planilha='Setor',
    colunas_intervalo='B:H',
    pula_linhas=2,
    numero_linhas=370
)

print("=" * 70)
print("DataFrame de Empresas e Setores da B3")
print("=" * 70)
print(f"\nDimensões do DataFrame: {df_b3_empresas_setor.shape}")
print(f"Linhas: {df_b3_empresas_setor.shape[0]} | Colunas: {df_b3_empresas_setor.shape[1]}")
print(f"\nColunas disponíveis:\n{df_b3_empresas_setor.columns.tolist()}")
print(f"\nÍndice (Códigos das Empresas):\n{df_b3_empresas_setor.index.tolist()[:10]}...")
print(f"\nPrimeiras linhas do DataFrame:")
print(df_b3_empresas_setor.head())
print("=" * 70)


DataFrame de Empresas e Setores da B3

Dimensões do DataFrame: (369, 6)
Linhas: 369 | Colunas: 6

Colunas disponíveis:
['BEEST', 'SETOR ECONÔMICO', 'SUBSETOR', 'SEGMENTO', 'NOME DE PREGÃO', 'SEGMENTO DE NEGOCIAÇÃO']

Índice (Códigos das Empresas):
['AZTE', 'BRAV', 'CSAN', 'RPMG', 'PETR', 'RECV', 'PRIO', 'RAIZ', 'UGPA', 'LUPA']...

Primeiras linhas do DataFrame:
            BEEST                  SETOR ECONÔMICO  \
CÓDIGO                                               
AZTE        Outra  Petróleo, Gás e Biocombustíveis   
BRAV        Outra  Petróleo, Gás e Biocombustíveis   
CSAN        Outra  Petróleo, Gás e Biocombustíveis   
RPMG        Outra  Petróleo, Gás e Biocombustíveis   
PETR    Escolhida  Petróleo, Gás e Biocombustíveis   

                               SUBSETOR                           SEGMENTO  \
CÓDIGO                                                                       
AZTE    Petróleo, Gás e Biocombustíveis  Exploração, Refino e Distribuição   
BRAV    Petróleo, Gás e

In [5]:
"""
# Arquivo Excel de origem da relação de papéis e setores
arquivo = "../utils/gera_tikers/B3_Empresas_Setor_20260206.xlsx"

# Planilha e aba onde estão os papéis e setores
planilha = 'Setor'
colunas_intervalo = 'B:H'
pula_linhas = 2
numero_linhas = 370

# DataFrame para armazenar os dados coletados
df_b3_empresas_setor = pd.read_excel(
                        io=arquivo, 
                        sheet_name=planilha,
                        usecols=colunas_intervalo,
                        skiprows=pula_linhas,
                        nrows=numero_linhas,
                        index_col='CÓDIGO' # Definir a coluna 'CÓDIGO' como índice do DataFrame
                        )

# Exibir o DataFrame para verificar os dados coletados
df_b3_empresas_setor.head()
"""

'\n# Arquivo Excel de origem da relação de papéis e setores\narquivo = "../utils/gera_tikers/B3_Empresas_Setor_20260206.xlsx"\n\n# Planilha e aba onde estão os papéis e setores\nplanilha = \'Setor\'\ncolunas_intervalo = \'B:H\'\npula_linhas = 2\nnumero_linhas = 370\n\n# DataFrame para armazenar os dados coletados\ndf_b3_empresas_setor = pd.read_excel(\n                        io=arquivo, \n                        sheet_name=planilha,\n                        usecols=colunas_intervalo,\n                        skiprows=pula_linhas,\n                        nrows=numero_linhas,\n                        index_col=\'CÓDIGO\' # Definir a coluna \'CÓDIGO\' como índice do DataFrame\n                        )\n\n# Exibir o DataFrame para verificar os dados coletados\ndf_b3_empresas_setor.head()\n'

In [6]:
# Extrair os nomes das colunas do DataFrame
colunas = df_b3_empresas_setor.columns.tolist()

# Exibir os nomes das colunas para verificar a estrutura do DataFrame
print(colunas)

['BEEST', 'SETOR ECONÔMICO', 'SUBSETOR', 'SEGMENTO', 'NOME DE PREGÃO', 'SEGMENTO DE NEGOCIAÇÃO']


In [7]:
# Símbolos base dos papais negociado na B3
simbolos_base = df_b3_empresas_setor.index.tolist()

# Exibir os símbolos base para verificar os dados coletados
print(simbolos_base)

['AZTE', 'BRAV', 'CSAN', 'RPMG', 'PETR', 'RECV', 'PRIO', 'RAIZ', 'UGPA', 'LUPA', 'OPCT', 'OSXB', 'VBBR', 'AURA', 'BRAP', 'CBAV', 'CMIN', 'LTEL', 'LTLA', 'VALE', 'FESA', 'GGBR', 'GOAU', 'CSNA', 'USIM', 'HAGA', 'MGEL', 'PATI', 'TKNO', 'PMAM', 'BRKM', 'DEXP', 'FHER', 'NUTR', 'VITT', 'CRPG', 'UNIP', 'DXCO', 'EUCA', 'KLBN', 'MSPA', 'NEMO', 'SUZB', 'RANI', 'SNSY', 'ETER', 'PTBL', 'AZEV', 'SOND', 'ARML', 'MILS', 'PRNR', 'EMBJ', 'FRAS', 'POMO', 'RAPT', 'RCSL', 'RSUL', 'TUPY', 'MWET', 'SHUL', 'WEGE', 'EALT', 'AERI', 'BDLL', 'INEP', 'KEPL', 'FRIO', 'PTCA', 'ROMI', 'MTSA', 'TASA', 'AZUL', 'GOLL', 'VSPT', 'MRSA', 'RAIL', 'HBSA', 'LOGN', 'LUXM', 'JSLG', 'SEQL', 'TGMA', 'CRTE', 'ECOR', 'MOTV', 'TPIS', 'AGRU', 'HMOB', 'IVPR', 'PSVM', 'BBML', 'CTAX', 'DTCY', 'ALPK', 'GGPS', 'VLID', 'MMAQ', 'RBNS', 'WLMM', 'TTEN', 'AGXY', 'APTI', 'SOJA', 'AGRO', 'CTCA', 'EGGY', 'SLCE', 'LAND', 'JALL', 'SMTO', 'BAUH', 'FICT', 'JBSS', 'MBRF', 'BEEF', 'MNPR', 'CAML', 'JOPA', 'MDIA', 'ODER', 'ABEV', 'ESPA', 'NATU', 'BOBR',

In [8]:
# Consulta na API do Yahoo Finance quais são os tickers correspondentes aos símbolos base
tickers = []
for simbolo in simbolos_base:
    ticker = yf.Ticker(simbolo + '.SA')  # Adiciona o sufixo '.SA' para ações da B3
    tickers.append(ticker)
# Exibir os tickers para verificar os dados coletados
print(tickers)


[yfinance.Ticker object <AZTE.SA>, yfinance.Ticker object <BRAV.SA>, yfinance.Ticker object <CSAN.SA>, yfinance.Ticker object <RPMG.SA>, yfinance.Ticker object <PETR.SA>, yfinance.Ticker object <RECV.SA>, yfinance.Ticker object <PRIO.SA>, yfinance.Ticker object <RAIZ.SA>, yfinance.Ticker object <UGPA.SA>, yfinance.Ticker object <LUPA.SA>, yfinance.Ticker object <OPCT.SA>, yfinance.Ticker object <OSXB.SA>, yfinance.Ticker object <VBBR.SA>, yfinance.Ticker object <AURA.SA>, yfinance.Ticker object <BRAP.SA>, yfinance.Ticker object <CBAV.SA>, yfinance.Ticker object <CMIN.SA>, yfinance.Ticker object <LTEL.SA>, yfinance.Ticker object <LTLA.SA>, yfinance.Ticker object <VALE.SA>, yfinance.Ticker object <FESA.SA>, yfinance.Ticker object <GGBR.SA>, yfinance.Ticker object <GOAU.SA>, yfinance.Ticker object <CSNA.SA>, yfinance.Ticker object <USIM.SA>, yfinance.Ticker object <HAGA.SA>, yfinance.Ticker object <MGEL.SA>, yfinance.Ticker object <PATI.SA>, yfinance.Ticker object <TKNO.SA>, yfinance.Tick

In [9]:
# Consulta na API do Yahoo Finance os dados históricos de um ticker específico
ticker_exemplo = 'PETR4.SA'  # Exemplo com o primeiro ticker da lista

dados_historicos = yf.download(ticker_exemplo, start=data_marco, end=data_atual)

dados_historicos.head()  # Exibir os dados históricos para verificar a coleta

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,PETR4.SA,PETR4.SA,PETR4.SA,PETR4.SA,PETR4.SA
Date,,,,,
2023-01-02,12.971099,13.474776,12.903187,13.321976,78424700
2023-01-03,12.642861,13.072968,12.524016,12.982419,96750300
2023-01-04,13.044669,13.350271,12.354236,12.427806,129504000
2023-01-05,13.514389,13.604939,13.101261,13.208788,73886000
2023-01-06,13.435160,13.763399,13.327633,13.548346,51851500


In [10]:
# Consulta na API do Yahoo Finance as informações gerais de um ticker específico
ticker_info = 'PETR11.SA'

yf.Ticker(ticker_exemplo).info

{'address1': 'Avenida Henrique Valadares, 28',
 'city': 'Rio De Janeiro',
 'state': 'RJ',
 'zip': '20231-030',
 'country': 'Brazil',
 'phone': '55 21 3224 2401',
 'website': 'https://petrobras.com.br',
 'industry': 'Oil & Gas Integrated',
 'industryKey': 'oil-gas-integrated',
 'industryDisp': 'Oil & Gas Integrated',
 'sector': 'Energy',
 'sectorKey': 'energy',
 'sectorDisp': 'Energy',
 'longBusinessSummary': 'Petróleo Brasileiro S.A. - Petrobras explores, produces, and sells oil and gas in Brazil and internationally. It operates through three segments: Exploration and Production; Refining, Transportation & Marketing; and Gas & Low Carbon Energies. The Exploration and Production segment explores, develops, and produces crude oil, natural gas liquids, and natural gas primarily for supplies to the domestic refineries. The Refining, Transportation and Marketing segment engages in the refining, logistics, transport, acquisition, and export of crude oil; trading of oil products; and producti

In [11]:
# Buscar todos os tickers da API do Yahoo Finance que começam com os símbolos base com sequenciais de papeis ON, PN e UNITS e armazenar uma lista de tickers correspondentes
tickers_correspondentes = []
for simbolo in simbolos_base:
    ticker_on = simbolo + '3.SA'  # Ações ordinárias (ON)
    ticker_pn = simbolo + '4.SA'  # Ações preferenciais (PN)
    ticker_units = simbolo + '11.SA'  # Units
    tickers_correspondentes.extend([ticker_on, ticker_pn, ticker_units])
# Exibir os tickers correspondentes para verificar os dados coletados
print(len(tickers_correspondentes))
print(tickers_correspondentes)


1107
['AZTE3.SA', 'AZTE4.SA', 'AZTE11.SA', 'BRAV3.SA', 'BRAV4.SA', 'BRAV11.SA', 'CSAN3.SA', 'CSAN4.SA', 'CSAN11.SA', 'RPMG3.SA', 'RPMG4.SA', 'RPMG11.SA', 'PETR3.SA', 'PETR4.SA', 'PETR11.SA', 'RECV3.SA', 'RECV4.SA', 'RECV11.SA', 'PRIO3.SA', 'PRIO4.SA', 'PRIO11.SA', 'RAIZ3.SA', 'RAIZ4.SA', 'RAIZ11.SA', 'UGPA3.SA', 'UGPA4.SA', 'UGPA11.SA', 'LUPA3.SA', 'LUPA4.SA', 'LUPA11.SA', 'OPCT3.SA', 'OPCT4.SA', 'OPCT11.SA', 'OSXB3.SA', 'OSXB4.SA', 'OSXB11.SA', 'VBBR3.SA', 'VBBR4.SA', 'VBBR11.SA', 'AURA3.SA', 'AURA4.SA', 'AURA11.SA', 'BRAP3.SA', 'BRAP4.SA', 'BRAP11.SA', 'CBAV3.SA', 'CBAV4.SA', 'CBAV11.SA', 'CMIN3.SA', 'CMIN4.SA', 'CMIN11.SA', 'LTEL3.SA', 'LTEL4.SA', 'LTEL11.SA', 'LTLA3.SA', 'LTLA4.SA', 'LTLA11.SA', 'VALE3.SA', 'VALE4.SA', 'VALE11.SA', 'FESA3.SA', 'FESA4.SA', 'FESA11.SA', 'GGBR3.SA', 'GGBR4.SA', 'GGBR11.SA', 'GOAU3.SA', 'GOAU4.SA', 'GOAU11.SA', 'CSNA3.SA', 'CSNA4.SA', 'CSNA11.SA', 'USIM3.SA', 'USIM4.SA', 'USIM11.SA', 'HAGA3.SA', 'HAGA4.SA', 'HAGA11.SA', 'MGEL3.SA', 'MGEL4.SA', 'MGEL11.

In [ ]:
# Verificar quais tickers correspondentes existem na API do Yahoo Finance e armazenar os tickers válidos em uma nova lista
tickers_validos = []
for ticker in tickers_correspondentes:
    try:
        yf.Ticker(ticker).info  # Verificar se o ticker existe na API do Yahoo Finance
        tickers_validos.append(ticker)  # Adicionar o ticker válido à lista
    except Exception as e:
        print(f"Ticker {ticker} não encontrado: {e}")  # Imprimir mensagem de erro para tickers não encontrados

In [ ]:
# Exibir os tickers válidos para verificar os dados coletados
print(len(tickers_validos))
print(tickers_validos)

-----------------------------------------------

#### Função para gerar os tickers a partir de um arquivo Excel

In [12]:
# Biblioteca para validar os tickers usando a API do Yahoo Finance
import yfinance as yf
from datetime import datetime, timedelta
import pandas as pd
import warnings

In [13]:
# Função para gerar a lista de tickers do Yahoo Finance
def gerar_tickers_yf(caminho_arquivo, planilha='Setor', colunas_intervalo='B:H', 
                    pula_linhas=2, numero_linhas=370):
    """
    Gera uma lista de tickers do Yahoo Finance a partir de um arquivo Excel com dados da B3.
    
    Esta função lê um arquivo Excel contendo informações de empresas listadas na B3
    e gera tickers correspondentes para as três classes de ações: Ordinárias (ON),
    Preferenciais (PN) e Units. Os tickers gerados seguem o padrão do Yahoo Finance.
    
    Parâmetros
    ----------
    caminho_arquivo : str
        Caminho completo ou relativo do arquivo Excel contendo dados das empresas e setores.
    planilha : str, optional
        Nome da aba/planilha no arquivo Excel (padrão: 'Setor').
    colunas_intervalo : str, optional
        Intervalo de colunas a ler no formato Excel (padrão: 'B:H').
        Exemplo: 'A:E' ou 'B:H'.
    pula_linhas : int, optional
        Número de linhas iniciais a pular na leitura (padrão: 2).
        Útil para ignorar cabeçalhos ou informações adicionais.
    numero_linhas : int, optional
        Número máximo de linhas de dados a ler (padrão: 370).
    
    Returno
    -------
    list
        Lista contendo tickers do Yahoo Finance no formato 'CÓDIGO#.SA', onde:
        - CÓDIGO#3.SA : Ações Ordinárias (ON)
        - CÓDIGO#4.SA : Ações Preferenciais (PN)
        - CÓDIGO#11.SA : Units
    
    Notas
    -----
    - O arquivo Excel deve conter uma coluna chamada 'CÓDIGO' que será usada como índice.
    - Cada símbolo base gera 3 tickers (ON, PN e Units).
    - O sufixo '.SA' é adicionado automaticamente para indicar ações da bolsa brasileira no Yahoo Finance.
    """
    # DataFrame para armazenar os dados coletados
    df_b3_empresas_setor = pd.read_excel(
                            io=caminho_arquivo, 
                            sheet_name=planilha,
                            usecols=colunas_intervalo,
                            skiprows=pula_linhas,
                            nrows=numero_linhas,
                            index_col='CÓDIGO'
                            )

    # Símbolos base dos papeis negociados na B3
    simbolos_base_b3 = df_b3_empresas_setor.index.tolist()

    # Gerar os tickers correspondentes para cada símbolo base, 
    # considerando as diferentes classes de ações (ON, PN, Units)
    tickers_yf = []
    for simbolo in simbolos_base_b3:
        ticker_on = simbolo + '3.SA'  # Ações ordinárias (ON)
        ticker_pn = simbolo + '4.SA'  # Ações preferenciais (PN)
        ticker_units = simbolo + '11.SA'  # Units
        tickers_yf.extend([ticker_on, ticker_pn, ticker_units])
    
    return tickers_yf

In [15]:
# Localização do arquivo base de setores e empresas listadas na B3
caminho_arquivo_b3 = "../utils/gera_tikers/"
arquivo_acoes_b3 = "B3_Empresas_Setor_20260206.xlsx"
arquivo_completo = caminho_arquivo_b3 + arquivo_acoes_b3

# Gerar a lista de tickers do Yahoo Finance a partir do arquivo Excel
lista_tickers_yf = gerar_tickers_yf(arquivo_completo)

# Exibir a quantidade total de tickers gerados e a lista completa
print("==============================================================")
print("Gerando a lista de tickers do Yahoo Finance a partir do arquivo Excel...")
print(f"Arquivo Excel utilizado: {arquivo_completo}")
print(f"A quantidade total de tickers gerados é: {len(lista_tickers_yf)}")
print(lista_tickers_yf)
print("==============================================================")

Gerando a lista de tickers do Yahoo Finance a partir do arquivo Excel...
Arquivo Excel utilizado: ../utils/gera_tikers/B3_Empresas_Setor_20260206.xlsx
A quantidade total de tickers gerados é: 1107
['AZTE3.SA', 'AZTE4.SA', 'AZTE11.SA', 'BRAV3.SA', 'BRAV4.SA', 'BRAV11.SA', 'CSAN3.SA', 'CSAN4.SA', 'CSAN11.SA', 'RPMG3.SA', 'RPMG4.SA', 'RPMG11.SA', 'PETR3.SA', 'PETR4.SA', 'PETR11.SA', 'RECV3.SA', 'RECV4.SA', 'RECV11.SA', 'PRIO3.SA', 'PRIO4.SA', 'PRIO11.SA', 'RAIZ3.SA', 'RAIZ4.SA', 'RAIZ11.SA', 'UGPA3.SA', 'UGPA4.SA', 'UGPA11.SA', 'LUPA3.SA', 'LUPA4.SA', 'LUPA11.SA', 'OPCT3.SA', 'OPCT4.SA', 'OPCT11.SA', 'OSXB3.SA', 'OSXB4.SA', 'OSXB11.SA', 'VBBR3.SA', 'VBBR4.SA', 'VBBR11.SA', 'AURA3.SA', 'AURA4.SA', 'AURA11.SA', 'BRAP3.SA', 'BRAP4.SA', 'BRAP11.SA', 'CBAV3.SA', 'CBAV4.SA', 'CBAV11.SA', 'CMIN3.SA', 'CMIN4.SA', 'CMIN11.SA', 'LTEL3.SA', 'LTEL4.SA', 'LTEL11.SA', 'LTLA3.SA', 'LTLA4.SA', 'LTLA11.SA', 'VALE3.SA', 'VALE4.SA', 'VALE11.SA', 'FESA3.SA', 'FESA4.SA', 'FESA11.SA', 'GGBR3.SA', 'GGBR4.SA', '

###  Função para filtrar tickers válidos na consulta do Yahoo Finance
Critério baseado no texto **"HTTP Error 404"** para selecionar os tickers válidos no Yahoo Finance. A função irá retornar uma lista de tickers válidos para consulta, eliminando aqueles que não estão disponíveis na plataforma.

In [16]:
# Filtrar os tickers válidos, excluindo aqueles que geram erro 404 
# ou retornam informações vazias.
def filtrar_tickers_validos(lista_tickers_yf):
    """
    Recebe uma lista de tickers e retorna apenas os válidos para 
    o Yahoo Finance.
    Tickers que geram mensagem contendo "HTTP Error 404" são excluídos.
    
    Ordem de verificação:
    1. Captura de exceções HTTP 404 (prioridade)
    2. Validação de info não vazio
    3. Verificação de campos essenciais (symbol)
    
    Parâmetros
    ----------
    lista_tickers_yf : list
        Lista de tickers a serem validados.
    
    Retorno
    -------
    list, list
        - tickers_validos_yf: Lista de tickers válidos para o Yahoo Finance.
        - tickers_erro_404_yf: Lista de tickers que geraram erro 404 ou retornaram informações vazias.
    """
    # Listas para armazenar os tickers válidos e os que geraram erro 404
    tickers_validos_yf = []
    tickers_erro_404_yf = []
    
    # Iterar sobre cada ticker e realizar as verificações
    for ticker in lista_tickers_yf:
        try:
            warnings.filterwarnings("ignore")  # Ignorar avisos de depreciação ou outros tipos de avisos
            info = yf.Ticker(ticker).info
            
        except Exception as e:
            # PRIORIDADE 1: Capturar e verificar primeiro se é um erro 404
            erro_str = str(e)
            
            if "HTTP Error 404" in erro_str or "Not Found" in erro_str:
                tickers_erro_404_yf.append(ticker)
                print(f"Ticker {ticker} gerou erro 404: {erro_str[:80]}")
            else:
                # Outros erros também são considerados inválidos
                tickers_erro_404_yf.append(ticker)
                print(f"Ticker {ticker} gerou erro: {erro_str[:80]}")
            continue
        
        # PRIORIDADE 2: Se não houve exceção, verificar se info é válido
        if not info or len(info) == 0:
            tickers_erro_404_yf.append(ticker)
            print(f"Ticker {ticker} retornou informações vazias.")
            continue
        
        # PRIORIDADE 3: Verificar se contém campos essenciais
        if 'symbol' not in info or info.get('symbol') is None:
            tickers_erro_404_yf.append(ticker)
            print(f"Ticker {ticker} não contém símbolo válido.")
            continue
        
        # Se passou por todas as verificações, o ticker é válido
        tickers_validos_yf.append(ticker)
        print(f"Ticker {ticker} é válido.")

    return tickers_validos_yf, tickers_erro_404_yf


In [17]:
'''
tickers_testes = ['PETR4.SA', 'VALE3.SA', 'AZTE3.SA', 'AZTE4.SA', 'AZTE11.SA', 'BRAV3.SA', 'BRAV4.SA', 'BRAV11.SA', 'CSAN3.SA', 'CSAN4.SA', 'CSAN11.SA', 'RPMG3.SA', 'RPMG4.SA', 'RPMG11.SA', 'PETR3.SA', 'PETR4.SA', 'PETR11.SA', 'RECV3.SA', 'RECV4.SA', 'RECV11.SA', 'PRIO3.SA', 'PRIO4.SA', 'PRIO11.SA', 'RAIZ3.SA', 'RAIZ4.SA', 'RAIZ11.SA', 'UGPA3.SA', 'UGPA4.SA', 'UGPA11.SA','ABCD3.SA']  # Exemplo de tickers para teste
print("A quantidade de tickers para o teste é:", len(tickers_testes))
'''

'\ntickers_testes = [\'PETR4.SA\', \'VALE3.SA\', \'AZTE3.SA\', \'AZTE4.SA\', \'AZTE11.SA\', \'BRAV3.SA\', \'BRAV4.SA\', \'BRAV11.SA\', \'CSAN3.SA\', \'CSAN4.SA\', \'CSAN11.SA\', \'RPMG3.SA\', \'RPMG4.SA\', \'RPMG11.SA\', \'PETR3.SA\', \'PETR4.SA\', \'PETR11.SA\', \'RECV3.SA\', \'RECV4.SA\', \'RECV11.SA\', \'PRIO3.SA\', \'PRIO4.SA\', \'PRIO11.SA\', \'RAIZ3.SA\', \'RAIZ4.SA\', \'RAIZ11.SA\', \'UGPA3.SA\', \'UGPA4.SA\', \'UGPA11.SA\',\'ABCD3.SA\']  # Exemplo de tickers para teste\nprint("A quantidade de tickers para o teste é:", len(tickers_testes))\n'

In [18]:
# Filtrar os tickers válidos, excluindo aqueles que geram erro 404 ou retornam informações vazias
tickers_validos_yf, tickers_erro_404_yf = filtrar_tickers_validos(lista_tickers_yf)

# Exibir a quantidade de tickers válidos e os que geraram erro 404
print("==============================================================")
print(f"A quantidade de tickers válidos é: {len(tickers_validos_yf)}")
print(f"Tickers válidos: {tickers_validos_yf}")
print()
print(f"A quantidade de tickers que geraram erro 404 ou informações vazias é: {len(tickers_erro_404_yf)}")
print(f"Tickers que geraram erro 404 ou informações vazias: {tickers_erro_404_yf}")
print("==============================================================")


Ticker AZTE3.SA é válido.


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: AZTE4.SA"}}}


Ticker AZTE4.SA não contém símbolo válido.
Ticker AZTE11.SA é válido.
Ticker BRAV3.SA é válido.


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: BRAV4.SA"}}}


Ticker BRAV4.SA não contém símbolo válido.


...

Ticker OPSE11.SA não contém símbolo válido.
Ticker OPTS3.SA é válido.


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: OPTS4.SA"}}}


Ticker OPTS4.SA não contém símbolo válido.


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: OPTS11.SA"}}}


Ticker OPTS11.SA não contém símbolo válido.
A quantidade de tickers válidos é: 505
Tickers válidos: ['AZTE3.SA', 'AZTE11.SA', 'BRAV3.SA', 'CSAN3.SA', 'RPMG3.SA', 'RPMG4.SA', 'PETR3.SA', 'PETR4.SA', 'RECV3.SA', 'PRIO3.SA', 'RAIZ4.SA', 'UGPA3.SA', 'UGPA4.SA', 'LUPA3.SA', 'LUPA11.SA', 'OPCT3.SA', 'OSXB3.SA', 'VBBR3.SA', 'BRAP3.SA', 'BRAP4.SA', 'CBAV3.SA', 'CMIN3.SA', 'LTLA3.SA', 'VALE3.SA', 'FESA3.SA', 'FESA4.SA', 'GGBR3.SA', 'GGBR4.SA', 'GOAU3.SA', 'GOAU4.SA', 'CSNA3.SA', 'USIM3.SA', 'HAGA3.SA', 'HAGA4.SA', 'MGEL3.SA', 'MGEL4.SA', 'PATI3.SA', 'PATI4.SA', 'TKNO3.SA', 'TKNO4.SA', 'PMAM3.SA', 'PMAM4.SA', 'PMAM11.SA', 'BRKM3.SA', 'DEXP3.SA', 'DEXP4.SA', 'FHER3.SA', 'NUTR3.SA', 'VITT3.SA', 'CRPG3.SA', 'UNIP3.SA', 'DXCO3.SA', 'EUCA3.SA', 'EUCA4.SA', 'KLBN3.SA', 'KLBN4.SA', 'KLBN11.SA', 'MSPA3.SA', 'MSPA4.SA', 'NEMO4.SA', 'SUZB3.SA', 'RANI3.SA', 'RANI4.SA', 'SNSY3.SA', 'ETER3.SA', 'ETER4.SA', 'PTBL3.SA', 'PTBL4.SA', 'AZEV3.SA', 'AZEV4.SA', 'AZEV11.SA', 'SOND3.SA', 'ARML3.SA', 'MILS3.SA', 'PRNR3

### Gerar arquivo CSV para viabilizar a consulta dos símbolos resultantes no Yahoo Finance

Sequência de instrução:
- Ler o arquivo Excel "B3_Empresas_Setor_20260206.xlsx", transformado em um Dataframe usando o Pandas.
- Utilizar o 4 caracteres iniciais dos Tickers Válidos "ticker_validos_yf" para buscar as informações na coluna "CÓDIGO" para obter as outras informações das colunas do Dataframe resultante do "B3_Empresas_Setor_20260206.xlsx".
- Gerar um arquivo csv com as informações combinadas das duas consultas:
    - coluna index: Ticker com valores correspondentes do tickers_validos_yf
    - colunas BEEST, SETOR ECONÔMICO, SBSETOR, SEGMENTO, NOME DE PREGÃO, SEGMENTO DE NEGOCIAÇÃO dos valores das respectivas colunas do df_b3_empresas_setor



In [19]:
# Exibir os tickers válidos
print("Tickers válidos para consulta no Yahoo Finance:")
print("Quantidade de tickers válidos: ", len(tickers_validos_yf))
print(tickers_validos_yf)

Tickers válidos para consulta no Yahoo Finance:
Quantidade de tickers válidos:  505
['AZTE3.SA', 'AZTE11.SA', 'BRAV3.SA', 'CSAN3.SA', 'RPMG3.SA', 'RPMG4.SA', 'PETR3.SA', 'PETR4.SA', 'RECV3.SA', 'PRIO3.SA', 'RAIZ4.SA', 'UGPA3.SA', 'UGPA4.SA', 'LUPA3.SA', 'LUPA11.SA', 'OPCT3.SA', 'OSXB3.SA', 'VBBR3.SA', 'BRAP3.SA', 'BRAP4.SA', 'CBAV3.SA', 'CMIN3.SA', 'LTLA3.SA', 'VALE3.SA', 'FESA3.SA', 'FESA4.SA', 'GGBR3.SA', 'GGBR4.SA', 'GOAU3.SA', 'GOAU4.SA', 'CSNA3.SA', 'USIM3.SA', 'HAGA3.SA', 'HAGA4.SA', 'MGEL3.SA', 'MGEL4.SA', 'PATI3.SA', 'PATI4.SA', 'TKNO3.SA', 'TKNO4.SA', 'PMAM3.SA', 'PMAM4.SA', 'PMAM11.SA', 'BRKM3.SA', 'DEXP3.SA', 'DEXP4.SA', 'FHER3.SA', 'NUTR3.SA', 'VITT3.SA', 'CRPG3.SA', 'UNIP3.SA', 'DXCO3.SA', 'EUCA3.SA', 'EUCA4.SA', 'KLBN3.SA', 'KLBN4.SA', 'KLBN11.SA', 'MSPA3.SA', 'MSPA4.SA', 'NEMO4.SA', 'SUZB3.SA', 'RANI3.SA', 'RANI4.SA', 'SNSY3.SA', 'ETER3.SA', 'ETER4.SA', 'PTBL3.SA', 'PTBL4.SA', 'AZEV3.SA', 'AZEV4.SA', 'AZEV11.SA', 'SOND3.SA', 'ARML3.SA', 'MILS3.SA', 'PRNR3.SA', 'EMBJ3.SA'

In [20]:
# Arquivo Excel de origem da relação de papéis e setores
arquivo = "../utils/gera_tikers/B3_Empresas_Setor_20260206.xlsx"

# Planilha e aba onde estão os papéis e setores
planilha = 'Setor'
colunas_intervalo = 'B:H'
pula_linhas = 2
numero_linhas = 370

# DataFrame para armazenar os dados coletados
df_b3_empresas_setor = pd.read_excel(
                        io=arquivo, 
                        sheet_name=planilha,
                        usecols=colunas_intervalo,
                        skiprows=pula_linhas,
                        nrows=numero_linhas,
                        index_col='CÓDIGO' # Definir a coluna 'CÓDIGO' como índice do DataFrame
                        )

# Exibir o DataFrame para verificar os dados coletados
df_b3_empresas_setor.head()

,BEEST,SETOR ECONÔMICO,SUBSETOR,SEGMENTO,NOME DE PREGÃO,SEGMENTO DE NEGOCIAÇÃO
CÓDIGO,,,,,,
AZTE,Outra,"Petróleo, Gás e Biocombustíveis","Petróleo, Gás e Biocombustíveis","Exploração, Refino e Distribuição",AZT ENERGIA,Básico Bolsa
BRAV,Outra,"Petróleo, Gás e Biocombustíveis","Petróleo, Gás e Biocombustíveis","Exploração, Refino e Distribuição",BRAVA,Novo Mercado
CSAN,Outra,"Petróleo, Gás e Biocombustíveis","Petróleo, Gás e Biocombustíveis","Exploração, Refino e Distribuição",COSAN,Novo Mercado
RPMG,Outra,"Petróleo, Gás e Biocombustíveis","Petróleo, Gás e Biocombustíveis","Exploração, Refino e Distribuição",PET MANGUINH,Básico Bolsa
PETR,Escolhida,"Petróleo, Gás e Biocombustíveis","Petróleo, Gás e Biocombustíveis","Exploração, Refino e Distribuição",PETROBRAS,Nível 2


In [22]:
# Iterar sobre os 4 primeiros caracteres dos tickers válidos para buscar as das colunas do DataFrame para gerar um novo DataFrame

# Criar uma lista para armazenar os dicionários de informações dos tickers válidos
tickers_info_list = []

# Iterar sobre os tickers válidos e extrair as informações correspondentes do DataFrame original
for ticker in tickers_validos_yf:
    # Extrair os 4 primeiros caracteres do ticker para buscar no DataFrame
    t = ticker[:4]
    # Localizar o ticker no DataFrame Empresas Setor da B3 e extrair as informações correspondentes
    info = df_b3_empresas_setor.loc[df_b3_empresas_setor.index.str.startswith(t)]
    # Para cada linha encontrada, adicionar o ticker como uma nova coluna
    for _, row in info.iterrows(): # Iterar sobre as linhas encontradas para o ticker
        row_dict = row.to_dict() # Converter a linha para um dicionário
        row_dict['TICKER'] = ticker # Adicionar o ticker ao dicionário
        tickers_info_list.append(row_dict) # Adicionar o dicionário à lista de informações dos tickers

# Criar o DataFrame a partir da lista
tickers_info = pd.DataFrame(tickers_info_list)

# Definir a coluna TICKER como index do DataFrame
tickers_info = tickers_info.set_index('TICKER')

# Exibir a quantidade de tickers_info gerados
print("Quantidade de tickers_info gerados: ", len(tickers_info))

# Exibir o DataFrame com as informações dos tickers válidos
tickers_info.head()

Quantidade de tickers_info gerados:  505


,BEEST,SETOR ECONÔMICO,SUBSETOR,SEGMENTO,NOME DE PREGÃO,SEGMENTO DE NEGOCIAÇÃO
TICKER,,,,,,
AZTE3.SA,Outra,"Petróleo, Gás e Biocombustíveis","Petróleo, Gás e Biocombustíveis","Exploração, Refino e Distribuição",AZT ENERGIA,Básico Bolsa
AZTE11.SA,Outra,"Petróleo, Gás e Biocombustíveis","Petróleo, Gás e Biocombustíveis","Exploração, Refino e Distribuição",AZT ENERGIA,Básico Bolsa
BRAV3.SA,Outra,"Petróleo, Gás e Biocombustíveis","Petróleo, Gás e Biocombustíveis","Exploração, Refino e Distribuição",BRAVA,Novo Mercado
CSAN3.SA,Outra,"Petróleo, Gás e Biocombustíveis","Petróleo, Gás e Biocombustíveis","Exploração, Refino e Distribuição",COSAN,Novo Mercado
RPMG3.SA,Outra,"Petróleo, Gás e Biocombustíveis","Petróleo, Gás e Biocombustíveis","Exploração, Refino e Distribuição",PET MANGUINH,Básico Bolsa


In [23]:
# Transformar o DataFrame tickers_info em um arquivo csv para ser utilizado posteriormente
tickers_info.to_csv('../utils/gera_tikers/tickers_info.csv', index=True) # Index True para manter a coluna TICKER como index no arquivo csv

### Função para gerar o arquivo CSV para consulta dos símbolos resultantes no Yahoo Finance
A função `gerar_csv_tickers_validos` tem como objetivo criar um arquivo CSV contendo os tickers válidos para consulta no Yahoo Finance, juntamente com informações adicionais obtidas a partir do arquivo Excel "B3_Empresas_Setor_20260206.xlsx". O processo envolve a leitura do arquivo Excel, a filtragem dos tickers válidos, e a combinação das informações para gerar um arquivo CSV estruturado.

In [24]:
def gerar_df_b3_empresas_setor(caminho_arquivo: str, 
                                    planilha: str = 'Setor', 
                                    colunas_intervalo: str = 'B:H', 
                                    pula_linhas: int = 2, 
                                    numero_linhas: int = 370) -> pd.DataFrame:
    """
    Carrega e retorna um DataFrame com as informações de empresas e setores da B3.
    
    Esta função lê um arquivo Excel contendo dados de empresas listadas na Bolsa
    de Valores do Brasil (B3), com informações sobre setores econômicos,
    subsetores, segmentos de negociação e outros dados estruturais.
    
    Parâmetros
    ----------
    caminho_arquivo : str
        Caminho completo ou relativo do arquivo Excel contendo dados das empresas e setores.
        Exemplo: '../utils/gera_tikers/B3_Empresas_Setor_20260206.xlsx'
    
    planilha : str, optional
        Nome da aba/planilha no arquivo Excel (padrão: 'Setor').
        A planilha deve conter uma coluna chamada 'CÓDIGO' que será usada como índice.
    
    colunas_intervalo : str, optional
        Intervalo de colunas a ler no formato Excel (padrão: 'B:H').
        Exemplos válidos: 'A:E', 'B:H', 'C:F'.
    
    pula_linhas : int, optional
        Número de linhas iniciais a pular na leitura (padrão: 2).
        Útil para ignorar cabeçalhos ou informações adicionais no início do arquivo.
    
    numero_linhas : int, optional
        Número máximo de linhas de dados a ler (padrão: 370).
        Se o arquivo tiver menos linhas, todas serão lidas.
    
    Retorno
    -------
    pd.DataFrame
        DataFrame com a coluna 'CÓDIGO' definida como índice, contendo as seguintes
        informações estruturais:
        - BEEST (ou similar)
        - SETOR ECONÔMICO
        - SUBSETOR
        - SEGMENTO
        - NOME DE PREGÃO
        - SEGMENTO DE NEGOCIAÇÃO
        
        O índice do DataFrame conterá os códigos das empresas (ex: 'PETR', 'VALE', 'ITUB').
    """
    try:
        # Ler o arquivo Excel com os parâmetros especificados
        df_b3_empresas_setor = pd.read_excel(
            io=caminho_arquivo,
            sheet_name=planilha,
            usecols=colunas_intervalo,
            skiprows=pula_linhas,
            nrows=numero_linhas,
            index_col='CÓDIGO'  # Definir a coluna 'CÓDIGO' como índice do DataFrame
        )
        
        return df_b3_empresas_setor
    
    # Tratamento de exceções para garantir que erros sejam informativos
    except FileNotFoundError:
        raise FileNotFoundError(f"Arquivo não encontrado: {caminho_arquivo}")
    except ValueError as e:
        raise ValueError(f"Erro ao ler o arquivo Excel: {str(e)}")
    except Exception as e:
        raise Exception(f"Erro inesperado ao processar o arquivo: {str(e)}")


# Gerar o DataFrame de empresas e setores da B3
caminho_arquivo_b3 = "../utils/gera_tikers/B3_Empresas_Setor_20260206.xlsx"

df_b3_empresas_setor = gerar_df_b3_empresas_setor(
    caminho_arquivo=caminho_arquivo_b3,
    planilha='Setor',
    colunas_intervalo='B:H',
    pula_linhas=2,
    numero_linhas=370
)

In [25]:
# Exibir o DataFrame 
df_b3_empresas_setor.head()

,BEEST,SETOR ECONÔMICO,SUBSETOR,SEGMENTO,NOME DE PREGÃO,SEGMENTO DE NEGOCIAÇÃO
CÓDIGO,,,,,,
AZTE,Outra,"Petróleo, Gás e Biocombustíveis","Petróleo, Gás e Biocombustíveis","Exploração, Refino e Distribuição",AZT ENERGIA,Básico Bolsa
BRAV,Outra,"Petróleo, Gás e Biocombustíveis","Petróleo, Gás e Biocombustíveis","Exploração, Refino e Distribuição",BRAVA,Novo Mercado
CSAN,Outra,"Petróleo, Gás e Biocombustíveis","Petróleo, Gás e Biocombustíveis","Exploração, Refino e Distribuição",COSAN,Novo Mercado
RPMG,Outra,"Petróleo, Gás e Biocombustíveis","Petróleo, Gás e Biocombustíveis","Exploração, Refino e Distribuição",PET MANGUINH,Básico Bolsa
PETR,Escolhida,"Petróleo, Gás e Biocombustíveis","Petróleo, Gás e Biocombustíveis","Exploração, Refino e Distribuição",PETROBRAS,Nível 2


In [26]:
def gerar_csv_tickers_info(df_b3_empresas_setor, tickers_validos, caminho_saida_csv=None):
    """
    Gera um DataFrame com informações da B3 para os tickers válidos do Yahoo Finance
    e, opcionalmente, salva o resultado em um arquivo CSV.

    A função usa o DataFrame `df_b3_empresas_setor` (com índice em 'CÓDIGO') e
    combina as informações para cada ticker válido, utilizando os 4 primeiros
    caracteres do ticker como chave de busca.

    Parâmetros
    ----------
    df_b3_empresas_setor : pd.DataFrame
        DataFrame com informações da B3 e índice na coluna 'CÓDIGO'.
    tickers_validos : list
        Lista de tickers válidos do Yahoo Finance (ex: 'AZTE3.SA', 'BRAV3.SA').
    caminho_saida_csv : str, optional
        Caminho para salvar o CSV. Se None, não salva (padrão: None).

    Retorno
    -------
    pd.DataFrame
        DataFrame com tickers como índice e informações da B3 como colunas.
    """
    # Criar lista para armazenar dados
    tickers_info_list = []
    
    # Iterar sobre os tickers válidos
    for ticker in tickers_validos:
        # Extrair os 4 primeiros caracteres do ticker
        t = ticker[:4]
        # Localizar informações correspondentes no DataFrame
        info = df_b3_empresas_setor.loc[df_b3_empresas_setor.index.str.startswith(t)]
        # Para cada linha encontrada, adicionar o ticker como nova coluna
        for _, row in info.iterrows():
            row_dict = row.to_dict()
            row_dict['TICKER'] = ticker
            tickers_info_list.append(row_dict)
    
    # Criar DataFrame a partir da lista de dicionários
    tickers_info = pd.DataFrame(tickers_info_list)
    
    # Definir TICKER como índice do DataFrame
    tickers_info = tickers_info.set_index('TICKER')
    
    # Salvar como CSV se caminho foi fornecido
    if caminho_saida_csv:
        tickers_info.to_csv(caminho_saida_csv, index=True)
        print(f"Arquivo CSV salvo em: {caminho_saida_csv}")
    
    return tickers_info

In [27]:
# Gerar o DataFrame com informações dos tickers válidos e salvar como CSV
caminho_saida_csv = '../utils/gera_tikers/tickers_info.csv'
tickers_info = gerar_csv_tickers_info(df_b3_empresas_setor, tickers_validos_yf, caminho_saida_csv)

# Exibir o DataFrame com as informações dos tickers válidos
print("DataFrame com informações dos tickers válidos:")
print(tickers_info.head())

Arquivo CSV salvo em: ../utils/gera_tikers/tickers_info.csv
DataFrame com informações dos tickers válidos:
           BEEST                  SETOR ECONÔMICO  \
TICKER                                              
AZTE3.SA   Outra  Petróleo, Gás e Biocombustíveis   
AZTE11.SA  Outra  Petróleo, Gás e Biocombustíveis   
BRAV3.SA   Outra  Petróleo, Gás e Biocombustíveis   
CSAN3.SA   Outra  Petróleo, Gás e Biocombustíveis   
RPMG3.SA   Outra  Petróleo, Gás e Biocombustíveis   

                                  SUBSETOR                           SEGMENTO  \
TICKER                                                                          
AZTE3.SA   Petróleo, Gás e Biocombustíveis  Exploração, Refino e Distribuição   
AZTE11.SA  Petróleo, Gás e Biocombustíveis  Exploração, Refino e Distribuição   
BRAV3.SA   Petróleo, Gás e Biocombustíveis  Exploração, Refino e Distribuição   
CSAN3.SA   Petróleo, Gás e Biocombustíveis  Exploração, Refino e Distribuição   
RPMG3.SA   Petróleo, Gás e Biocombu